## Data Cleaning & Pre-processing

In [16]:

import pandas as pd
import numpy as np
import re  
from sklearn.preprocessing import MultiLabelBinarizer

pd.set_option('display.max_columns', None)

In [17]:
raw_data_path = '../data/game_data.csv' 
df = pd.read_csv(raw_data_path)

print(f"Starting shape: {df.shape}")
df.head(3)

Starting shape: (89618, 47)


,appid,name,release_date,required_age,price,dlc_count,detailed_description,about_the_game,short_description,reviews,header_image,website,support_url,support_email,windows,mac,linux,metacritic_score,metacritic_url,achievements,recommendations,notes,supported_languages,full_audio_languages,packages,developers,publishers,categories,genres,screenshots,movies,user_score,score_rank,positive,negative,estimated_owners,average_playtime_forever,average_playtime_2weeks,median_playtime_forever,median_playtime_2weeks,discount,peak_ccu,tags,pct_pos_total,num_reviews_total,pct_pos_recent,num_reviews_recent
0,730,Counter-Strike 2,2012-08-21,0,0.0,1,"For over two decades, Counter-Strike has offer...","For over two decades, Counter-Strike has offer...","For over two decades, Counter-Strike has offer...",NaN,https://shared.akamai.steamstatic.com/store_it...,http://counter-strike.net/,NaN,NaN,True,False,True,0,NaN,1,4401572,Includes intense violence and blood.,"['Czech', 'Danish', 'Dutch', 'English', 'Finni...","['English', 'Indonesian']","[{'title': 'Buy Counter-Strike 2', 'descriptio...",['Valve'],['Valve'],"['Multi-player', 'Cross-Platform Multiplayer',...","['Action', 'Free To Play']",['https://shared.akamai.steamstatic.com/store_...,['http://video.akamai.steamstatic.com/store_tr...,0,NaN,7480813,1135108,100000000 - 200000000,33189,879,5174,350,0,1212356,"{'FPS': 90857, 'Shooter': 65397, 'Multiplayer'...",86,8632939,82,96473
1,578080,PUBG: BATTLEGROUNDS,2017-12-21,0,0.0,0,"LAND, LOOT, SURVIVE! Play PUBG: BATTLEGROUNDS ...","LAND, LOOT, SURVIVE! Play PUBG: BATTLEGROUNDS ...",Play PUBG: BATTLEGROUNDS for free. Land on str...,NaN,https://shared.akamai.steamstatic.com/store_it...,https://www.pubg.com,https://support.pubg.com/hc/en-us,NaN,True,False,False,0,NaN,37,1732007,NaN,"['English', 'Korean', 'Simplified Chinese', 'F...",[],[],['PUBG Corporation'],"['KRAFTON, Inc.']","['Multi-player', 'PvP', 'Online PvP', 'Stats',...","['Action', 'Adventure', 'Massively Multiplayer...",['https://shared.akamai.steamstatic.com/store_...,[],0,NaN,1487960,1024436,50000000 - 100000000,0,0,0,0,0,616738,"{'Survival': 14838, 'Shooter': 12727, 'Battle ...",59,2513842,68,16720
2,570,Dota 2,2013-07-09,0,0.0,2,"The most-played game on Steam. Every day, mill...","The most-played game on Steam. Every day, mill...","Every day, millions of players worldwide enter...",“A modern multiplayer masterpiece.” 9.5/10 – D...,https://shared.akamai.steamstatic.com/store_it...,http://www.dota2.com/,NaN,NaN,True,True,True,90,https://www.metacritic.com/game/pc/dota-2?ftag...,0,14337,"Dota 2 includes fantasy violence, use of alcoh...","['Bulgarian', 'Czech', 'Danish', 'Dutch', 'Eng...","['English', 'Korean', 'Simplified Chinese', 'V...","[{'title': 'Buy Dota 2', 'description': '', 's...",['Valve'],['Valve'],"['Multi-player', 'Co-op', 'Steam Trading Cards...","['Action', 'Strategy', 'Free To Play']",['https://shared.akamai.steamstatic.com/store_...,['http://video.akamai.steamstatic.com/store_tr...,0,NaN,1998462,451338,200000000 - 500000000,43031,1536,898,892,0,555977,"{'Free to Play': 59933, 'MOBA': 20158, 'Multip...",81,2452595,80,29366


In [18]:
# 1. Size
print(df.shape)

(89618, 47)


In [19]:
# 2. What are we looking at
print(df.head())

    appid                             name release_date  required_age  price  \
0     730                 Counter-Strike 2   2012-08-21             0   0.00   
1  578080              PUBG: BATTLEGROUNDS   2017-12-21             0   0.00   
2     570                           Dota 2   2013-07-09             0   0.00   
3  271590        Grand Theft Auto V Legacy   2015-04-13            17   0.00   
4  359550  Tom Clancy's Rainbow Six® Siege   2015-12-01            17   3.99   

   dlc_count                               detailed_description  \
0          1  For over two decades, Counter-Strike has offer...   
1          0  LAND, LOOT, SURVIVE! Play PUBG: BATTLEGROUNDS ...   
2          2  The most-played game on Steam. Every day, mill...   
3          0  When a young street hustler, a retired bank ro...   
4          9  Edition Comparison Ultimate Edition The Tom Cl...   

                                      about_the_game  \
0  For over two decades, Counter-Strike has offer...   
1  L

In [20]:
# 3. What are the data types and null counts
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 89618 entries, 0 to 89617
Data columns (total 47 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   appid                     89618 non-null  int64  
 1   name                      89618 non-null  str    
 2   release_date              89618 non-null  str    
 3   required_age              89618 non-null  int64  
 4   price                     89618 non-null  float64
 5   dlc_count                 89618 non-null  int64  
 6   detailed_description      89421 non-null  str    
 7   about_the_game            89398 non-null  str    
 8   short_description         89498 non-null  str    
 9   reviews                   10401 non-null  str    
 10  header_image              89618 non-null  str    
 11  website                   41114 non-null  str    
 12  support_url               44110 non-null  str    
 13  support_email             78798 non-null  str    
 14  windows          

In [21]:
# 4. Are there any obvious statistical weirdness
print(df.describe())

              appid  required_age         price     dlc_count  \
count  8.961800e+04  89618.000000  89618.000000  89618.000000   
mean   1.656904e+06      0.183624      7.309623      0.595583   
std    9.168390e+05      1.725594     13.331073     15.351920   
min    2.000000e+01     -1.000000      0.000000      0.000000   
25%    8.550525e+05      0.000000      0.990000      0.000000   
50%    1.524730e+06      0.000000      4.990000      0.000000   
75%    2.430852e+06      0.000000      9.990000      0.000000   
max    3.542350e+06     21.000000    999.980000   3427.000000   

       metacritic_score  achievements  recommendations    user_score  \
count      89618.000000  89618.000000     8.961800e+04  89618.000000   
mean           2.903245     20.552333     1.009401e+03      0.032817   
std           14.445358    163.562418     2.204815e+04      1.615149   
min            0.000000      0.000000     0.000000e+00      0.000000   
25%            0.000000      0.000000     0.000000e+00

In [22]:
# 5. How many unique genres/developers are there
print(df['genres'].nunique())

2689


### Cleaning

In [24]:
# Midpoint of Estimated Owners
def parse_owners(owner_str):
    """Extracts numbers from a string and returns the midpoint."""
    if pd.isna(owner_str):
        return 0
        
    # Remove commas and extract all continuous numbers
    clean_str = str(owner_str).replace(',', '')
    numbers = re.findall(r'\d+', clean_str)
    
    if len(numbers) == 2:
        return (int(numbers[0]) + int(numbers[1])) / 2
    elif len(numbers) == 1:
        return int(numbers[0])
    return 0

# Apply the function and create a clean numeric column
df['owners_clean'] = df['estimated_owners'].apply(parse_owners)

# Drop the old messy column
df.drop('estimated_owners', axis=1, inplace=True)

In [25]:
# Convert 'Free' or 'free' to '0.0', then strip any '$' signs
df['price'] = df['price'].astype(str).str.lower().replace('free', '0.0')
df['price'] = df['price'].str.replace('$', '', regex=False)

# Force conversion to numeric. If it hits weird text it can't parse, it turns it to NaN
df['price'] = pd.to_numeric(df['price'], errors='coerce')

# Fill any games that errored out or were missing with a price of 0
df['price'] = df['price'].fillna(0.0)

In [ ]:
# Drop rows where 'name' or 'developers' is completely blank
df.dropna(subset=['name', 'developers'], inplace=True)

# For reviews and playtime, fill NaNs with 0 instead of dropping
cols_to_fill_zero = ['positive', 'negative', 'average_forever', 'median_forever']
for col in cols_to_fill_zero:
    if col in df.columns:
        df[col] = df[col].fillna(0)

In [32]:
import ast

# Only run if the 'genres' column still exists
if 'genres' in df.columns:
    # 1. Fill missing and parse string-lists into real lists
    df['genres'] = df['genres'].fillna('[]').apply(ast.literal_eval)

    # 2. Use MultiLabelBinarizer
    mlb = MultiLabelBinarizer()
    genre_encoded = mlb.fit_transform(df['genres'])

    # 3. Create DataFrame and concatenate
    genre_df = pd.DataFrame(genre_encoded, columns=[f"genre_{c}" for c in mlb.classes_])
    df = pd.concat([df.reset_index(drop=True), genre_df.reset_index(drop=True)], axis=1)
    
    # Drop ONLY if it exists
    df.drop('genres', axis=1, inplace=True)

# Always show the results
print(df.filter(like='genre_').head())


   genre_360 Video  genre_Accounting  genre_Action  genre_Adventure  \
0                0                 0             1                0   
1                0                 0             1                1   
2                0                 0             1                0   
3                0                 0             1                1   
4                0                 0             1                0   

   genre_Animation & Modeling  genre_Audio Production  genre_Casual  \
0                           0                       0             0   
1                           0                       0             0   
2                           0                       0             0   
3                           0                       0             0   
4                           0                       0             0   

   genre_Design & Illustration  genre_Documentary  genre_Early Access  \
0                            0                  0                   0   


In [37]:
# Replace the -1 sentinel with NaN for both percentages and counts
cols_with_sentinels = [
    'pct_pos_total', 
    'pct_pos_recent', 
    'num_reviews_total', 
    'num_reviews_recent'
]

for col in cols_with_sentinels:
    df[col] = df[col].replace(-1, float('nan'))

In [34]:
df['release_date'] = pd.to_datetime(df['release_date'])

In [35]:
# Convert booleans to 1 or 0
platform_cols = ['windows', 'mac', 'linux']
df[platform_cols] = df[platform_cols].astype(int)


In [38]:
# Save the cleaned DataFrame to a new CSV file
df.to_csv('../data/game_data_cleaned.csv', index=False)
